# Free Hit backtest - 2025/26

`backtest.ipynb` asks whether each model predicts the right *amount*. This asks
the only question that actually pays off: if you had played a Free Hit in a
given gameweek, what would the tool have picked, and how would it have done?

For every gameweek from 4 onwards it rebuilds player state from earlier
gameweeks only, projects, solves the full squad under the real constraints -
£100.0m, fifteen players, three per club, a legal formation - and then looks up
what those players actually scored.

Runtime is about a minute and a half.

### The benchmark, and why it is a proxy

FPL's published weekly average isn't available for a past season - the API
carries `average_entry_score` only for the season in progress, and no archive
republishes it once the season rolls over.

What is available is `selected`, the number of managers owning each player,
recorded per gameweek. So the crowd's team can be rebuilt directly: the
most-owned legal eleven, captained by the most-owned player. That's the
`template` column.

It's a proxy, and a generous one - it reads somewhat above the true FPL
average, since it never carries an injured player, never takes a −4 hit, and
always captains the popular pick. Beating it is therefore a meaningfully harder
test than beating the published average would be.

Two other reference points are reported for scale:

- **`hindsight`** - the same optimiser handed the actual results. This is a
  ceiling, not a target: nothing that forecasts can approach it, and the gap to it
  is mostly irreducible variance rather than model error.
- **`mean_player_pts`** - the average score of every player who featured, which
  is roughly what picking at random returns.

In [1]:
import sys, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "fplfh").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

import numpy as np
import pandas as pd

from fplfh.evaluate import backtest_free_hit

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

SEASON = "2025-26"
START_GW = 4          # first gameweek with enough history to project from
BUDGET = 100.0        # a real Free Hit is squad value + bank; 100.0 is the baseline

## 1. Run it

In [2]:
t0 = time.time()
results = backtest_free_hit(SEASON, start_gw=START_GW, budget=BUDGET)
print(f"\ndone in {time.time() - t0:.0f}s over {len(results)} gameweeks")

  GW4   xP  67.6   scored    71   template    83   ceiling   149   (C) Ismaïla Sarr


  GW5   xP  57.8   scored    59   template    56   ceiling   135   (C) Bruno Borges Fer


  GW6   xP  68.9   scored    64   template    47   ceiling   160   (C) Erling Haaland


  GW7   xP  71.4   scored    74   template    64   ceiling   151   (C) Erling Haaland


  GW8   xP  63.7   scored    85   template    65   ceiling   186   (C) Erling Haaland


...

  GW34  xP  53.8   scored    55   template    57   ceiling   136   (C) Bruno Borges Fer


  GW35  xP  63.2   scored    50   template    58   ceiling   153   (C) Dominic Calvert-


  GW36  xP  91.1   scored    62   template    74   ceiling   154   (C) Erling Haaland


  GW37  xP  62.0   scored    54   template    75   ceiling   154   (C) Bruno Borges Fer


  GW38  xP  57.8   scored    59   template    28   ceiling   150   (C) Erling Haaland

done in 41s over 35 gameweeks


## 2. How did it do?

`actual` is the XI plus the doubled captain - what the Free Hit would have scored.

In [3]:
r = results
print(f"gameweeks tested            : {len(r)}  (GW{r.event.min()}-{r.event.max()})")
print()
print(f"  Free Hit squad, mean score : {r.actual.mean():6.1f}   (sd {r.actual.std():.1f})")
print(f"  template XI, mean score    : {r.template.mean():6.1f}   (sd {r.template.std():.1f})")
print(f"  perfect hindsight ceiling  : {r.hindsight.mean():6.1f}")
print()
print(f"  mean margin over template  : {r.vs_template.mean():+6.1f} points per gameweek")
print(f"  gameweeks beating template : {(r.vs_template > 0).sum()} of {len(r)}"
      f"  ({(r.vs_template > 0).mean():.0%})")
print()
print(f"  share of the achievable gap closed: "
      f"{(r.actual.mean() - r.template.mean()) / (r.hindsight.mean() - r.template.mean()):.1%}")

gameweeks tested            : 35  (GW4-38)

  Free Hit squad, mean score :   61.0   (sd 14.3)
  template XI, mean score    :   53.9   (sd 16.2)
  perfect hindsight ceiling  :  154.9

  mean margin over template  :   +7.1 points per gameweek
  gameweeks beating template : 22 of 35  (63%)

  share of the achievable gap closed: 7.0%


### Is the margin real, or noise?

Thirty-five gameweeks is not many, and weekly scores are volatile. A paired test
on the per-gameweek margin says whether the difference is distinguishable from
zero. It is a *paired* comparison - both teams play the same fixtures in the same
week - which removes most of the week-to-week variance.

In [4]:
from scipy import stats

diff = r.vs_template.dropna()
t, p = stats.ttest_1samp(diff, 0)
w = stats.wilcoxon(diff)
se = diff.std(ddof=1) / np.sqrt(len(diff))

print(f"  mean margin      : {diff.mean():+.2f} points per gameweek")
print(f"  standard error   : {se:.2f}")
print(f"  95% CI           : {diff.mean() - 1.96 * se:+.2f} to {diff.mean() + 1.96 * se:+.2f}")
print(f"  paired t-test    : t = {t:.2f}, p = {p:.3f}")
print(f"  Wilcoxon signed  : p = {w.pvalue:.3f}   (no normality assumption)")
print()
print("  Over a 35-gameweek season that margin compounds to roughly "
      f"{diff.mean() * 38:+.0f} points,")
print("  but read the confidence interval before taking that seriously.")

  mean margin      : +7.11 points per gameweek
  standard error   : 3.04
  95% CI           : +1.16 to +13.07
  paired t-test    : t = 2.34, p = 0.025
  Wilcoxon signed  : p = 0.042   (no normality assumption)

  Over a 35-gameweek season that margin compounds to roughly +270 points,
  but read the confidence interval before taking that seriously.


## 3. Was the projection itself calibrated?

Separate question from whether the team was good. `xp` is what the model expected
the chosen XI to score; `actual` is what it did. If the optimiser is systematically
picking players whose expected points are overstated - which is exactly what an
optimiser does, since it selects on the estimate and therefore on its error - this
is where it shows.

In [5]:
print(f"  mean projected : {r.xp.mean():6.2f}")
print(f"  mean realised  : {r.actual.mean():6.2f}")
print(f"  bias           : {r.xp.mean() - r.actual.mean():+6.2f} points per gameweek")
print(f"  correlation    : {r.xp.corr(r.actual):6.3f}")
print(f"  mean abs error : {(r.xp - r.actual).abs().mean():6.2f}")

  mean projected :  65.13
  mean realised  :  60.97
  bias           :  +4.16 points per gameweek
  correlation    :  0.294
  mean abs error :  11.55


## 4. Gameweek by gameweek

In [6]:
cols = ["event", "xp", "actual", "template", "vs_template", "hindsight",
        "captain", "captain_pts", "formation", "cost"]
print(r[cols].to_string(index=False))

 event     xp  actual  template  vs_template  hindsight                captain  captain_pts formation   cost
     4  67.63   71.00     83.00       -12.00     149.00           Ismaïla Sarr         0.00     4-3-3 100.00
     5  57.79   59.00     56.00         3.00     135.00 Bruno Borges Fernandes        10.00     3-5-2  99.60
     6  68.94   64.00     47.00        17.00     160.00         Erling Haaland        16.00     3-5-2  99.80
     7  71.37   74.00     64.00        10.00     151.00         Erling Haaland         8.00     3-5-2 100.00
     8  63.68   85.00     65.00        20.00     186.00         Erling Haaland        13.00     3-5-2  99.30
     9  65.30   52.00     62.00       -10.00     170.00         Erling Haaland         2.00     3-5-2  99.90
    10  66.06   93.00     70.00        23.00     141.00         Erling Haaland        13.00     3-4-3 100.00
    11  65.56   35.00     31.00         4.00     153.00         Erling Haaland         4.00     4-4-2 100.00
    12  63.99   41.

In [7]:
print("Best gameweeks against the template")
print(r.nlargest(5, "vs_template")[
    ["event", "actual", "template", "vs_template", "captain", "captain_pts"]
].to_string(index=False))
print()
print("Worst")
print(r.nsmallest(5, "vs_template")[
    ["event", "actual", "template", "vs_template", "captain", "captain_pts"]
].to_string(index=False))

Best gameweeks against the template
 event  actual  template  vs_template        captain  captain_pts
    13   69.00     24.00        45.00 Erling Haaland         2.00
    33   84.00     42.00        42.00 Erling Haaland        13.00
    20   67.00     28.00        39.00 Erling Haaland         2.00
    24   93.00     61.00        32.00 Enzo Fernández         8.00
    38   59.00     28.00        31.00 Erling Haaland         0.00

Worst
 event  actual  template  vs_template                captain  captain_pts
    25   52.00     79.00       -27.00         Jurriën Timber         6.00
    23   35.00     58.00       -23.00         Erling Haaland         1.00
    37   54.00     75.00       -21.00 Bruno Borges Fernandes         9.00
     4   71.00     83.00       -12.00           Ismaïla Sarr         0.00
    36   62.00     74.00       -12.00         Erling Haaland        11.00


## 5. The captain

On a Free Hit the captain is doubled, so one pick carries far more weight than any
other decision the optimiser makes. It is worth isolating.

In [8]:
print(f"  mean captain return (doubled) : {2 * r.captain_pts.mean():5.1f} points")
print(f"  share of the XI total         : {2 * r.captain_pts.sum() / r.actual.sum():.1%}")
print(f"  captain blanked (<=2 pts)     : {(r.captain_pts <= 2).sum()} of {len(r)}"
      f"  ({(r.captain_pts <= 2).mean():.0%})")
print(f"  captain hauled (>=10 pts)     : {(r.captain_pts >= 10).sum()} of {len(r)}"
      f"  ({(r.captain_pts >= 10).mean():.0%})")
print()
print("most-chosen captains")
print(r.captain.value_counts().head(8).to_string())

  mean captain return (doubled) :  13.1 points
  share of the XI total         : 21.6%
  captain blanked (<=2 pts)     : 11 of 35  (31%)
  captain hauled (>=10 pts)     : 11 of 35  (31%)

most-chosen captains
captain
Erling Haaland            18
Bruno Borges Fernandes     7
Jurriën Timber             2
Mohamed Salah              2
Ismaïla Sarr               1
Bukayo Saka                1
Florian Wirtz              1
Enzo Fernández             1


## 6. What this does and does not show

A Free Hit is a one-off chip. In a real season you play it once, in a gameweek
you choose - typically a blank or a double, which is precisely when the
optimiser's edge over a static template is largest. Averaging over all 35
gameweeks understates what the chip is worth when played deliberately, so the
gameweek-by-gameweek table is the more useful read for that.

The template is also generous. It never carries an injured player, never takes
a hit, and always captains the popular pick, so it reads above the FPL average
it stands in for - the margin against the true average would be larger.

Availability flags are missing historically, so the backtest fields players who
were actually injured. The live tool sees those flags, which cuts the other
way: real performance should be better than what's shown here.

And everything from `backtest.ipynb` section 7 still applies - most
importantly, every parameter in the model was fitted on this same season, so
these numbers are an upper bound.